In [ ]:
import json
with open("timings1.json", "r+") as f:
    timings = json.load(f)

In [ ]:
import numpy as np
names = set([t["molname"]for t in timings])
for basis in ["sto-3g", "6-31g"]:
    data = {n: {"JordanWigner":0, "ParityEncoding":0,"BravyiKitaev":0,"JKMN":0} for n in name}
    for name in names:
        mystr = ""
        for t in timings:
            if t["molname"]!=name:
                continue
            elif t["basis"] != basis:
                continue
            if t["encoding"] == "JordanWigner":
                mystr += r"\ce{"+ f"{name.upper()}"+ r"}" + f"& {basis.upper()} & {t["n_modes"]} & {t["n_terms"]}"
            mystr += f" & {float(np.mean(t["time"])):.2e} ({float(np.std(t["time"])):.2e})"
        print(mystr + r" \\")

In [ ]:
import json
with open("reductions.json", "r+") as f:
    timings = json.load(f)

In [ ]:
from turtle import bk
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.markers import MarkerStyle
timings.sort(key=lambda x: x["n_modes"])
fig,axs =plt.subplots(nrows=1,ncols=3, sharex=True,sharey=True, figsize=(12,3.5))
colors = {"JordanWigner":"tab:blue", "ParityEncoding":"tab:orange", "BravyiKitaev":"tab:green", "JKMN":"tab:red"}
markers = {'H2': 0,
 'LiH': 1,
 'H2O': 2,
 'BH3': 3,
 'CH4': 4,
 'N2': 5,
 'HCN': 6,
 'C2H2': 7,
 'CH3F': 8,
 'ethene': 9,
 'ozone': 10}
for name in ["JordanWigner", "ParityEncoding", "BravyiKitaev", "JKMN"]:
    for ind,basis in enumerate(["sto-3g", "6-31g","cc-pVDZ"]):
        this_encoding = [t for t in timings if name in t["encoding"] and t["basis"] == basis]
        modes = np.array([t["n_modes"] for t in this_encoding])
        marker_style.update(markeredgecolor="none", markersize=15)
        for te in this_encoding:
            pw = 1-float(te["topphatt_weights"][0])/float(te["naive_weights"][0])
            cpw = 1-float(te["topphatt_weights"][1])/float(te["naive_weights"][1])
            marker = te["molname"]
            axs[ind].scatter(pw, cpw, color = colors[name], marker=f"${markers[marker]}$")

# where some data has already been plotted to ax
handles, labels = plt.gca().get_legend_handles_labels()
# manually define a new patch 
jw_legend = mlines.Line2D([], [], label='Jodan-Wigner', marker="s", linestyle="", color="tab:blue", mec="k", markersize=8)
pe_legend = mlines.Line2D([], [], label='Parity', marker="s", linestyle="", color="tab:orange", mec="k", markersize=8)
bk_legend = mlines.Line2D([], [], label='Bravyi-Kitaev', marker="s", linestyle="", color="tab:green", mec="k", markersize=8)
jkmn_legend = mlines.Line2D([], [], label='JKMN', marker="s", linestyle="", color="tab:red", mec="k", markersize=8)
axs[0].set_title("STO-3G")
axs[1].set_title("6-31G")
axs[2].set_title("cc-pVDZ")
axs[0].grid()
axs[1].grid()
axs[2].grid()

# handles is a list, so append manual patch
handles.extend([
    jw_legend,
    pe_legend,
    bk_legend,
    jkmn_legend,
                ])

axs[1].set_xlabel("$W_{P}$ Reduction")
axs[0].set_ylabel("$W_{CP}$ Reduction")
axs[2].legend(handles=handles, ncols=1, loc="lower right")
plt.xlim(-0.1,0.5)
plt.ylim(-0.1,0.5)
plt.show()